In [6]:
import pyvista as pv
import numpy as np
import ezdxf
from pathlib import Path
import matplotlib.pyplot as plt

pv.set_jupyter_backend("html")


# ============================================================
# BASIC TERRAIN / GEOMETRY HELPERS
# ============================================================

def gaussian_hill(x, y, x0, y0, height, width):
    """
    Simple smooth hill centered at (x0, y0).

    height : maximum hill height
    width  : controls how wide/spread out the hill is
    """
    r2 = (x - x0)**2 + (y - y0)**2
    return height * np.exp(-r2 / (2 * width**2))


def terrain_z(x, y):
    """
    Simple synthetic terrain with small hills around the three support locations.
    """

    z = 0.0

    # Hill near vertical line 1
    z += gaussian_hill(
        x, y,
        x0=16.0,
        y0=-5.5,
        height=1.5,
        width=4.0,
    )

    # Larger hill near vertical line 2
    z += gaussian_hill(
        x, y,
        x0=2.0,
        y0=0.0,
        height=2.5,
        width=5.0,
    )

    # Hill near vertical line 3
    z += gaussian_hill(
        x, y,
        x0=-16.0,
        y0=0.0,
        height=1.5,
        width=4.0,
    )

    # random hill
    z += gaussian_hill(
        x, y,
        x0 = -8.0,
        y0 = 0.0,
        height = -2.5,
        width = 3.0
    )

    return z


def make_vertical_line(x, y, height):
    """
    Create a vertical PyVista line starting from the terrain surface.

    Parameters
    ----------
    x, y : float
        Plan-view coordinates of the vertical line.
    height : float
        Height of the vertical line above the terrain.

    Returns
    -------
    vertical_line : pyvista.PolyData
        The vertical line.
    bottom_point : tuple
        Bottom point of the line, located on the terrain.
    top_point : tuple
        Top point of the line.
    """

    z_surface = terrain_z(x, y)

    bottom_point = (x, y, z_surface)
    top_point = (x, y, z_surface + height)

    vertical_line = pv.Line(
        pointa=bottom_point,
        pointb=top_point,
    )

    return vertical_line, bottom_point, top_point


def make_catenary_between_points(point1, point2, a, n=200):
    """
    Create a PyVista polyline representing a catenary between two 3D points.

    The catenary is drawn in the vertical plane passing through point1 and point2.
    The horizontal span is the plan-view distance between the two points.
    """

    p1 = np.asarray(point1, dtype=float)
    p2 = np.asarray(point2, dtype=float)

    if a <= 0:
        raise ValueError("Catenary parameter 'a' must be positive.")

    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]

    S = np.sqrt(dx**2 + dy**2)

    if S == 0:
        raise ValueError("The two points must not have the same x,y coordinates.")

    ux = dx / S
    uy = dy / S

    h = p2[2] - p1[2]

    s = np.linspace(0.0, S, n)

    s_vertex = S / 2.0 - a * np.arcsinh(
        h / (2.0 * a * np.sinh(S / (2.0 * a)))
    )

    z_vertex = p1[2] - a * (np.cosh((0.0 - s_vertex) / a) - 1.0)

    z = z_vertex + a * (np.cosh((s - s_vertex) / a) - 1.0)

    x = p1[0] + ux * s
    y = p1[1] + uy * s

    points = np.column_stack([x, y, z])
    line = pv.lines_from_points(points)

    return line, points


# ============================================================
# PROFILE HELPERS
# ============================================================

def build_profile_from_catenary(catenary_points):
    """
        Build longitudinal profile data from 3D catenary points.

        The profile x-axis is chainage along the horizontal projection
        of the catenary alignment.

        Parameters
        ----------
        catenary_points : np.ndarray
            Array of shape (N, 3), with columns x, y, z.

        Returns
        -------
        profile : dict
            Dictionary containing:
                s           : horizontal chainage
                x           : original x coordinates
                y           : original y coordinates
                z_ground    : terrain elevation directly below catenary
                z_catenary  : catenary elevation
                clearance   : z_catenary - z_ground
    """

    points = np.asarray(catenary_points, dtype=float)

    x = points[:, 0]
    y = points[:, 1]
    z_catenary = points[:, 2]

    dx = np.diff(x)
    dy = np.diff(y)

    ds = np.sqrt(dx**2 + dy**2)
    s = np.insert(np.cumsum(ds), 0, 0.0)

    z_ground = terrain_z(x, y)
    clearance = z_catenary - z_ground

    profile = {
        "s": s,
        "x": x,
        "y": y,
        "z_ground": z_ground,
        "z_catenary": z_catenary,
        "clearance": clearance,
    }

    return profile


def plot_profile(profile, bottom_point1, top_point1, bottom_point2, top_point2, title):
    """
    Plot a 2D longitudinal profile using Matplotlib.

    The horizontal axis is chainage.
    The vertical axis is elevation.
    """

    s = profile["s"]

    plt.figure(figsize=(10, 4))

    plt.plot(s, profile["z_ground"], label="Ground profile")
    plt.plot(s, profile["z_catenary"], label="Catenary profile")

    # Support/profile vertical lines
    plt.plot(
        [s[0], s[0]],
        [bottom_point1[2], top_point1[2]],
        label="Support 1",
    )

    plt.plot(
        [s[-1], s[-1]],
        [bottom_point2[2], top_point2[2]],
        label="Support 2",
    )

    plt.xlabel("Chainage along alignment")
    plt.ylabel("Elevation z")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


# ============================================================
# DXF HELPERS
# ============================================================

def ensure_layer(doc, layer_name):
    """
    Create a DXF layer if it does not already exist.
    """

    if layer_name not in doc.layers:
        doc.layers.add(layer_name)


def export_profile_to_dxf(
    filename,
    profile,
    bottom_point1,
    top_point1,
    bottom_point2,
    top_point2,
):
    """
    Export a 2D longitudinal profile to DXF.

    The DXF coordinates are:
        X = chainage
        Y = elevation

    Exported entities:
        - ground profile polyline
        - catenary profile polyline
        - two vertical support lines
    """

    filename = Path(filename)
    filename.parent.mkdir(exist_ok=True)

    doc = ezdxf.new("R2010")
    msp = doc.modelspace()

    ensure_layer(doc, "PROFILE_GROUND")
    ensure_layer(doc, "PROFILE_CATENARY")
    ensure_layer(doc, "PROFILE_SUPPORTS")

    s = profile["s"]

    ground_points_2d = list(zip(s, profile["z_ground"]))
    catenary_points_2d = list(zip(s, profile["z_catenary"]))

    # Ground profile
    msp.add_lwpolyline(
        ground_points_2d,
        dxfattribs={"layer": "PROFILE_GROUND"},
    )

    # Catenary profile
    msp.add_lwpolyline(
        catenary_points_2d,
        dxfattribs={"layer": "PROFILE_CATENARY"},
    )

    # Support 1 in profile
    msp.add_line(
        (s[0], bottom_point1[2]),
        (s[0], top_point1[2]),
        dxfattribs={"layer": "PROFILE_SUPPORTS"},
    )

    # Support 2 in profile
    msp.add_line(
        (s[-1], bottom_point2[2]),
        (s[-1], top_point2[2]),
        dxfattribs={"layer": "PROFILE_SUPPORTS"},
    )

    doc.saveas(filename)

    print(f"2D profile DXF written to: {filename}")

def export_merged_profiles_to_dxf(
    filename,
    profiles_data,
):
    """
    Export multiple 2D longitudinal profiles into one continuous DXF.

    Each segment is drawn using:
        X = cumulative chainage
        Y = elevation

    The second segment starts where the first segment ends.
    The third segment starts where the second segment ends, etc.

    Parameters
    ----------
    filename : str
        Output DXF filename.

    profiles_data : list of dict
        Each dictionary should contain:
            name
            profile
            bottom_point1
            top_point1
            bottom_point2
            top_point2
    """

    filename = Path(filename)
    filename.parent.mkdir(exist_ok=True)

    doc = ezdxf.new("R2010")
    msp = doc.modelspace()

    ensure_layer(doc, "MERGED_GROUND")
    ensure_layer(doc, "MERGED_CATENARY")
    ensure_layer(doc, "MERGED_SUPPORTS")
    ensure_layer(doc, "MERGED_LABELS")

    current_offset = 0.0

    for i, item in enumerate(profiles_data):
        name = item["name"]
        profile = item["profile"]

        bottom_point1 = item["bottom_point1"]
        top_point1 = item["top_point1"]
        bottom_point2 = item["bottom_point2"]
        top_point2 = item["top_point2"]

        local_s = profile["s"]
        global_s = local_s + current_offset

        ground_points_2d = list(zip(global_s, profile["z_ground"]))
        catenary_points_2d = list(zip(global_s, profile["z_catenary"]))

        # Ground profile segment
        msp.add_lwpolyline(
            ground_points_2d,
            dxfattribs={"layer": "MERGED_GROUND"},
        )

        # Catenary profile segment
        msp.add_lwpolyline(
            catenary_points_2d,
            dxfattribs={"layer": "MERGED_CATENARY"},
        )

        # Draw the first support only for the first segment.
        # Otherwise, the shared support between segment 1 and segment 2
        # would be drawn twice.
        if i == 0:
            msp.add_line(
                (global_s[0], bottom_point1[2]),
                (global_s[0], top_point1[2]),
                dxfattribs={"layer": "MERGED_SUPPORTS"},
            )

        # Draw the ending support of every segment.
        # For segment 1, this is support 2.
        # For segment 2, this is support 3.
        # etc.
        msp.add_line(
            (global_s[-1], bottom_point2[2]),
            (global_s[-1], top_point2[2]),
            dxfattribs={"layer": "MERGED_SUPPORTS"},
        )

        # Optional label near the start of each segment
        # msp.add_text(
        #     name,
        #     dxfattribs={
        #         "layer": "MERGED_LABELS",
        #         "height": 0.35,
        #     },
        # ).set_placement((global_s[0], top_point1[2] + 0.5))

        # Advance the offset so the next segment starts here
        current_offset = global_s[-1]

    doc.saveas(filename)

    #print(f"Continuous merged 2D profile DXF written to: {filename}")

def export_3d_geometry_to_dxf(
    filename,
    vertical_segments,
    catenary_point_sets,
):
    """
    Export 3D support lines and 3D catenaries to DXF.

    Parameters
    ----------
    filename : str
        Output DXF filename.
    vertical_segments : list
        List of (bottom_point, top_point) pairs.
    catenary_point_sets : list
        List of catenary point arrays. Each array should be shape (N, 3).
    """

    filename = Path(filename)
    filename.parent.mkdir(exist_ok=True)

    doc = ezdxf.new("R2010")
    msp = doc.modelspace()

    ensure_layer(doc, "SUPPORTS_3D")
    ensure_layer(doc, "CATENARIES_3D")

    # Export vertical support lines
    for bottom_point, top_point in vertical_segments:
        msp.add_line(
            bottom_point,
            top_point,
            dxfattribs={"layer": "SUPPORTS_3D"},
        )

    # Export catenaries as 3D polylines
    for points in catenary_point_sets:
        points = np.asarray(points, dtype=float)
        dxf_points = [tuple(p) for p in points]

        msp.add_polyline3d(
            dxf_points,
            dxfattribs={"layer": "CATENARIES_3D"},
        )

    doc.saveas(filename)

    print(f"3D geometry DXF written to: {filename}")


# ============================================================
# TERRAIN
# ============================================================

x = np.linspace(-30, 30, 100)
y = np.linspace(-30, 30, 100)

xx, yy = np.meshgrid(x, y)
zz = terrain_z(xx, yy)

grid = pv.StructuredGrid(xx, yy, zz)
grid["height"] = zz.ravel(order="F")

contours = grid.contour(20, scalars="height")


# ============================================================
# VERTICAL LINES
# ============================================================

vertical_line1, bottom_point1, top_point1 = make_vertical_line(
    x=16.0,
    y=-5.5,
    height=5.0,
)

vertical_line2, bottom_point2, top_point2 = make_vertical_line(
    x= 2.0,
    y= 0.0,
    height=5.0,
)

vertical_line3, bottom_point3, top_point3 = make_vertical_line(
    x=-16.0,
    y=0,
    height=5.0,
)

#print(bottom_point1[2] - bottom_point3[2])

# ============================================================
# CATENARIES
# ============================================================

a1 = 30.0
a2 = 10.0

catenary_line1, catenary_points1 = make_catenary_between_points(
    top_point1,
    top_point2,
    a1,
    n=300,
)

catenary_line2, catenary_points2 = make_catenary_between_points(
    top_point2,
    top_point3,
    a2,
    n=300,
)


# ============================================================
# BUILD PROFILES
# ============================================================

profile1 = build_profile_from_catenary(catenary_points1)
profile2 = build_profile_from_catenary(catenary_points2)



# ============================================================
# EXPORT DXF FILES
# ============================================================


export_merged_profiles_to_dxf(
    filename="outputs/merged_profiles.dxf",
    profiles_data=[
        {
            "name": "LINE_1_TO_2",
            "profile": profile1,
            "bottom_point1": bottom_point1,
            "top_point1": top_point1,
            "bottom_point2": bottom_point2,
            "top_point2": top_point2,
        },
        {
            "name": "LINE_2_TO_3",
            "profile": profile2,
            "bottom_point1": bottom_point2,
            "top_point1": top_point2,
            "bottom_point2": bottom_point3,
            "top_point2": top_point3,
        },
    ],
)




# ============================================================
# 3D PYVISTA PLOT
# ============================================================

frustum_points = np.array([
    [-2, -2, 0], [2, -2, 0], [2, 2, 0], [-2, 2, 0],
    [-1, -1, 8], [1, -1, 8], [1, 1, 8], [-1, 1, 8],
])

frustum_edges = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4),
         (0,4), (1,5), (2,6), (3,7)]


p = pv.Plotter()

p.add_mesh(grid, opacity=0.6)
p.add_mesh(contours, line_width=3)

p.add_mesh(vertical_line1, line_width=2)
p.add_mesh(vertical_line2, line_width=2)
p.add_mesh(vertical_line3, line_width=2)

p.add_mesh(catenary_line1, line_width=4)
p.add_mesh(catenary_line2, line_width=4)

#for a, b in frustum_edges:
#    p.add_mesh(pv.Line(frustum_points[a], frustum_points[b]), line_width=4)

p.add_axes()
p.show()

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…